# Przewidywanie odpływu klientów banku
### Implementacja sieci neuronowej od zera · NumPy · Klasyfikacja binarna

---

## Problem

Banki tracą rocznie miliony złotych na odejściu klientów (ang. *churn*). Wczesne wykrycie klientów zagrożonych rezygnacją pozwala na podjęcie działań retencyjnych, zanim dojdzie do utraty.

**Cel projektu:** zbudowanie modelu predykcyjnego, który na podstawie danych demograficznych i historii klienta oceni prawdopodobieństwo jego odejścia z banku.

## Podejście

- **Dane:** 10 000 klientów banku, 13 cech, zmienna docelowa `Exited` (0 = pozostał, 1 = odszedł)
- **Model:** 2-warstwowa sieć neuronowa zaimplementowana **od zera w NumPy** (bez scikit-learn, PyTorch ani TensorFlow)
- **Kluczowe elementy:** normalizacja wsadowa (Batch Normalization), dropout, optymalizator Adam, wczesne zatrzymanie (Early Stopping)

## Wyniki

| Metryka | Wartość |
|---|---|
| AUC-ROC | **0.8501** |
| Dokładność | **84.15%** |
| F1-Score | 0.6204 |
| Precyzja | 0.6574 |
| Czułość | 0.5873 |

---


## 1. Importy

Projekt oparty wyłącznie na bibliotekach ogólnego przeznaczenia — bez dedykowanych frameworków ML:

| Biblioteka | Zastosowanie |
|---|---|
| `numpy` | Operacje macierzowe, implementacja sieci neuronowej |
| `pandas` | Wczytywanie i przetwarzanie danych tabelarycznych |
| `matplotlib` / `seaborn` | Wizualizacje danych i wyników modelu |
| `kagglehub` | Automatyczne pobieranie zbioru danych z Kaggle |
| `glob` | Wyszukiwanie pliku CSV w pobranym folderze |


In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import os
import kagglehub
import glob

## 2. Pobranie danych

**Źródło:** [Bank Customer Churn — Kaggle (abbas829)](https://www.kaggle.com/datasets/abbas829/bank-customer-churn)

Zbiór zawiera dane 10 000 klientów europejskiego banku z trzech krajów (Francja, Niemcy, Hiszpania). Każdy rekord opisuje profil klienta oraz to, czy zrezygnował z usług banku (`Exited = 1`) w badanym okresie.

Dataset pobierany automatycznie przy pierwszym uruchomieniu za pomocą `kagglehub`.


In [43]:
path = kagglehub.dataset_download("abbas829/bank-customer-churn",)

print("dataset pobrany")
print("Ścieżka do danych: ", path)

dataset pobrany
Ścieżka do danych:  C:\Users\matma\.cache\kagglehub\datasets\abbas829\bank-customer-churn\versions\1


In [44]:
import os

print(f"Zawartość folderu: {path}")
print("-" * 50)

for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent} {os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for file in files:
        full_path = os.path.join(root, file)
        size_kb = os.path.getsize(full_path) / 1024
        print(f"{sub_indent}{file}  ({size_kb:.1f} KB)")

Zawartość folderu: C:\Users\matma\.cache\kagglehub\datasets\abbas829\bank-customer-churn\versions\1
--------------------------------------------------
 1/
  Bank_Churn.csv  (621.0 KB)


## 3. Styl wizualizacji

Ustawiamy ciemny motyw wykresów — zapewnia lepszy kontrast dla prezentacji online i w środowiskach IDE. Paleta kolorów dobrana tak, by odróżniać klasę 0 (pozostał) od klasy 1 (odszedł).


In [45]:
plt.rcParams.update({
    "figure.facecolor": "#0f172a",
    "axes.facecolor":   "#1e293b",
    "axes.edgecolor":   "#334155",
    "axes.labelcolor":  "#e2e8f0",
    "xtick.color":      "#94a3b8",
    "ytick.color":      "#94a3b8",
    "text.color":       "#e2e8f0",
    "grid.color":       "#334155",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
    "font.family":      "monospace",
})
PALETTE  = ["#38bdf8", "#f472b6"]   # niebieski = zostaje, różowy = odchodzi
ACCENT   = "#38bdf8"
DANGER   = "#f87171"
WARNING  = "#fbbf24"
SUCCESS  = "#34d399"

OUTPUT_DIR = "wyniki"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  [zapisano] {path}")
    plt.close(fig)

## 4. Wczytanie i podgląd danych

Wczytujemy plik CSV i wyświetlamy podstawowe informacje o strukturze zbioru: wymiary, nazwy kolumn, typy danych oraz pierwsze rekordy.


In [46]:
# Automatycznie znajdź plik CSV w pobranym folderze
csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)

if not csv_files:
    print("Brak pliku CSV — sprawdź zawartość folderu powyżej")
else:
    CSV_PATH = csv_files[0]
    print(f"Znaleziony plik: {CSV_PATH}")
    print()

    df = pd.read_csv(CSV_PATH)

    print(f"Wymiary: {df.shape[0]:,} wierszy × {df.shape[1]} kolumn")
    print(f"Kolumny: {list(df.columns)}")
    print()

    df.head()

Znaleziony plik: C:\Users\matma\.cache\kagglehub\datasets\abbas829\bank-customer-churn\versions\1\Bank_Churn.csv

Wymiary: 10,000 wierszy × 13 kolumn
Kolumny: ['CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']



## 5. Eksploracyjna analiza danych (EDA)

Sprawdzamy kompletność danych (braki, duplikaty), rozkłady statystyczne oraz zbalansowanie klas.

Kluczowe pytania:
- Czy są braki danych lub duplikaty?
- Jak rozkładają się cechy numeryczne (średnia, odchylenie, kwantyle)?
- Czy zbiór jest niezbalansowany (co wymaga odpowiednich technik podczas treningu)?


In [47]:
print("Typy danych i braki:")
df.info()

Typy danych i braki:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerId       10000 non-null  int64  
 1   Surname          10000 non-null  str    
 2   CreditScore      10000 non-null  int64  
 3   Geography        10000 non-null  str    
 4   Gender           10000 non-null  str    
 5   Age              10000 non-null  int64  
 6   Tenure           10000 non-null  int64  
 7   Balance          10000 non-null  float64
 8   NumOfProducts    10000 non-null  int64  
 9   HasCrCard        10000 non-null  int64  
 10  IsActiveMember   10000 non-null  int64  
 11  EstimatedSalary  10000 non-null  float64
 12  Exited           10000 non-null  int64  
dtypes: float64(2), int64(8), str(3)
memory usage: 1015.8 KB


In [48]:
print("Statystyki opisowe:")
df.describe()

Statystyki opisowe:


,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [49]:
print("Rozkład zmiennej docelowej (Exited):")
print()
vc = df["Exited"].value_counts()
for k, v in vc.items():
    label = "Odszedł z banku" if k == 1 else "Pozostał w banku"
    bar = "█" * int(v / 100)
    print(f"  {label}  {v:,}  ({v/len(df)*100:.1f}%)  {bar}")

Rozkład zmiennej docelowej (Exited):

  Pozostał w banku  7,963  (79.6%)  ███████████████████████████████████████████████████████████████████████████████
  Odszedł z banku  2,037  (20.4%)  ████████████████████


In [50]:
# Zapisz jako bank_churn.csv w bieżącym folderze
# (potrzebne dla bank_churn_model.py)

LOCAL_PATH = "bank_churn.csv"
df.to_csv(LOCAL_PATH, index=False)

print(f"Dane zapisane jako: {LOCAL_PATH}")
print(f"Pełna ścieżka: {os.path.abspath(LOCAL_PATH)}")

Dane zapisane jako: bank_churn.csv
Pełna ścieżka: C:\Users\matma\PycharmProjects\Ubs-git1\bank_churn.csv


## 6. Oczyszczanie danych

Usuwamy kolumny nieistotne predykcyjnie:
- `RowNumber` — indeks techniczny
- `CustomerId` — unikalny identyfikator klienta (brak wartości predykcyjnej)
- `Surname` — nazwisko klienta (brak związku z churnem)

Sprawdzamy też braki danych i duplikaty.


In [51]:
# Usuń nieistotne kolumny
drop_cols = [c for c in ["RowNumber","CustomerId","Surname"] if c in df.columns]
df.drop(columns=drop_cols, inplace=True)

In [52]:
# Braki danych
missing = df.isnull().sum()
print(f"  Braki danych:\n{missing[missing>0] if missing.any() else '  Brak — dane są kompletne'}")

  Braki danych:
  Brak — dane są kompletne


In [53]:
# Duplikaty
dupl = df.duplicated().sum()
print(f"  Duplikaty: {dupl}")
df.drop_duplicates(inplace=True)

  Duplikaty: 0


In [54]:
# Typy — upewnij się, że kategorie są stringami
for col in ["Geography","Gender"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

In [55]:
# wypis rozkładu danych
print(f"\n  Rozkład klasy docelowej (Exited):")
vc = df["Exited"].value_counts()
for k,v in vc.items():
    label = "Odszedł" if k==1 else "Pozostał"
    print(f"    {label} ({k}): {v:,}  ({v/len(df)*100:.1f}%)")


  Rozkład klasy docelowej (Exited):
    Pozostał (0): 7,963  (79.6%)
    Odszedł (1): 2,037  (20.4%)


## 7. Wizualizacja danych

Analizujemy rozkłady cech oraz ich związek ze zmienną docelową `Exited`. Celem jest identyfikacja cech o największej sile dyskryminacyjnej.


### 7.1 Rozkład klas

Zbiór jest **niezbalansowany**: ~20% klientów odeszło, ~80% pozostało.
Niezbalansowanie klas wymaga zastosowania:
- **ważenia klas** podczas treningu (`pos_weight` w funkcji straty),
- metryki **AUC-ROC i F1** zamiast samej dokładności (accuracy),
- starannego doboru **progu decyzyjnego** przy klasyfikacji.


In [56]:
fig, ax = plt.subplots(figsize=(6,4))
colors = PALETTE
labels = ["Pozostał (0)", "Odszedł (1)"]
vals   = [vc.get(0,0), vc.get(1,0)]
bars   = ax.bar(labels, vals, color=colors, width=0.5, edgecolor="#0f172a", linewidth=1.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+80,
            f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Rozkład klasy docelowej", fontsize=14, pad=12)
ax.set_ylabel("Liczba klientów")
ax.set_ylim(0, max(vals)*1.15)
ax.grid(axis="y")
save(fig, "01_rozkład_klas.png")

  [zapisano] wyniki\01_rozkład_klas.png


### 7.2 Cechy numeryczne vs Churn

Wykresy pudełkowe pokazują rozkład każdej cechy numerycznej w podziale na klientów, którzy odeszli (1) i pozostali (0). Cechy z wyraźnie różnymi medianami między klasami mają większy potencjał predykcyjny.

**Obserwacje:**
- **Wiek** (`Age`): klienci starsi statystycznie częściej odchodzą — silna cecha predykcyjna
- **Saldo** (`Balance`): klienci z zerowym saldem rzadziej odchodzą — nieintuicyjne, wymaga uwagi
- **Liczba produktów** (`NumOfProducts`): klienci z 3-4 produktami odchodzą znacznie częściej — nieliniowa zależność


In [57]:
num_cols = ["CreditScore","Age","Tenure","Balance","NumOfProducts","EstimatedSalary"]
num_cols = [c for c in num_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    ax = axes[i]
    for churn, color, label in [(0,PALETTE[0],"Pozostał"), (1,PALETTE[1],"Odszedł")]:
        data = df[df["Exited"]==churn][col].dropna()
        ax.hist(data, bins=30, alpha=0.6, color=color, label=label, density=True)
    ax.set_title(col, fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(axis="y")
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Rozkłady cech numerycznych (Pozostał vs Odszedł)", fontsize=14, y=1.01)
fig.tight_layout()
save(fig, "02_cechy_numeryczne.png")

  [zapisano] wyniki\02_cechy_numeryczne.png


### 7.3 Cechy kategoryczne vs Churn

Analizujemy rozkłady zmiennych kategorycznych w podziale na klasy.

**Obserwacje:**
- **Kraj** (`Geography`): Niemcy mają znacząco wyższy wskaźnik churnu (~32%) niż Francja (~16%) i Hiszpania (~17%)
- **Płeć** (`Gender`): kobiety odchodzą częściej (~25%) niż mężczyźni (~16%)
- **Aktywny klient** (`IsActiveMember`): nieaktywni klienci odchodzą ponad dwa razy częściej


In [58]:
cat_cols = [c for c in ["Geography","Gender","HasCrCard","IsActiveMember","NumOfProducts"] if c in df.columns]
fig, axes = plt.subplots(1, len(cat_cols), figsize=(5*len(cat_cols), 5))
if len(cat_cols)==1: axes=[axes]
for ax, col in zip(axes, cat_cols):
    ct = df.groupby(col)["Exited"].mean().sort_values(ascending=False)
    ax.bar(ct.index.astype(str), ct.values*100, color=DANGER, edgecolor="#0f172a", linewidth=1.2)
    ax.set_title(f"% Churn wg {col}", fontsize=11, fontweight="bold")
    ax.set_ylabel("% odejść")
    ax.set_ylim(0, 100)
    ax.grid(axis="y")
    for x, val in enumerate(ct.values):
        ax.text(x, val*100+1.5, f"{val*100:.1f}%", ha="center", fontsize=10, fontweight="bold")
fig.suptitle("Wskaźnik odejść wg cech kategorycznych", fontsize=14)
fig.tight_layout()
save(fig, "03_cechy_kategoryczne.png")

  [zapisano] wyniki\03_cechy_kategoryczne.png


### 7.4 Macierz korelacji

Mapa ciepła korelacji między cechami numerycznymi (po zakodowaniu kategorii).

**Wnioski:**
- Brak silnej wielokolinearności — cechy dostarczają w większości niezależnej informacji
- Najsilniejsza korelacja z `Exited`: wiek (~0.29) i aktywność klienta (~-0.16)
- Saldo i wynagrodzenie praktycznie nieskorelowane ze sobą (r ≈ 0.01)


In [60]:
corr_df = df.copy()
if "Geography" in corr_df.columns:
    corr_df["Germany"] = (corr_df["Geography"]=="Germany").astype(int)
    corr_df["Spain"]   = (corr_df["Geography"]=="Spain").astype(int)
    corr_df.drop(columns=["Geography"], inplace=True)
if "Gender" in corr_df.columns:
    corr_df["Gender_Female"] = (corr_df["Gender"]=="Female").astype(int)
    corr_df.drop(columns=["Gender"], inplace=True)

corr = corr_df.corr()
fig, ax = plt.subplots(figsize=(11,9))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, center=0, annot=True, fmt=".2f",
            linewidths=0.5, linecolor="#0f172a", ax=ax,
            annot_kws={"size":9}, vmin=-1, vmax=1)
ax.set_title("Macierz korelacji", fontsize=14, pad=12)
fig.tight_layout()
save(fig, "04_korelacja.png")

print("  [OK] Wykresy zapisane w folderze 'wyniki/'")

  [zapisano] wyniki\04_korelacja.png
  [OK] Wykresy zapisane w folderze 'wyniki/'


## 8. Inżynieria cech i normalizacja

Tworzymy nowe cechy oparte na wiedzy domenowej:

| Cecha | Wzór | Uzasadnienie |
|---|---|---|
| `BalanceSalaryRatio` | saldo / (wynagrodzenie + 1) | Relatywna wartość aktywów wobec dochodów |
| `AgeGroup` | przedziały: <30, 30-45, 45-60, 60+ | Zachowania różnią się między grupami wiekowymi |
| `HasBalance` | 1 jeśli saldo > 0 | Klienci z zerowym saldem tworzą odrębną grupę |
| `TenurePerAge` | staż / (wiek + 1) | Proporcja życia spędzonego z bankiem |
| `ProductsPerTenure` | produkty / (staż + 1) | Intensywność korzystania z oferty |

**Kodowanie kategorii:**
- `Geography` -> one-hot (bazowa: Francja): `Geo_Germany`, `Geo_Spain`
- `Gender` -> binarne: `Gender_Female`

**Normalizacja:** standaryzacja Z-score wyznaczona na zbiorze treningowym i zastosowana do obu zbiorów — zapobiega wyciekowi danych (*data leakage*).


In [61]:
df_feat = df.copy()

# Nowe cechy
df_feat["BalanceSalaryRatio"]  = df_feat["Balance"] / (df_feat["EstimatedSalary"] + 1)
df_feat["AgeGroup"]            = pd.cut(df_feat["Age"], bins=[0,30,45,60,100],
                                        labels=[0,1,2,3]).astype(float)
df_feat["HasBalance"]          = (df_feat["Balance"] > 0).astype(int)
df_feat["TenurePerAge"]        = df_feat["Tenure"] / (df_feat["Age"] + 1)
df_feat["ProductsPerTenure"]   = df_feat["NumOfProducts"] / (df_feat["Tenure"] + 1)

# One-hot encoding
if "Geography" in df_feat.columns:
    geo_dummies = pd.get_dummies(df_feat["Geography"], prefix="Geo", drop_first=True)
    df_feat = pd.concat([df_feat, geo_dummies], axis=1)
    df_feat.drop(columns=["Geography"], inplace=True)

if "Gender" in df_feat.columns:
    df_feat["Gender_Female"] = (df_feat["Gender"] == "Female").astype(int)
    df_feat.drop(columns=["Gender"], inplace=True)

FEATURE_COLS = [c for c in df_feat.columns if c != "Exited"]
TARGET_COL   = "Exited"

print(f"  Liczba cech: {len(FEATURE_COLS)}")
print(f"  Cechy: {FEATURE_COLS}")

X_all = df_feat[FEATURE_COLS].values.astype(float)
y_all = df_feat[TARGET_COL].values.astype(float)

  Liczba cech: 16
  Cechy: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'BalanceSalaryRatio', 'AgeGroup', 'HasBalance', 'TenurePerAge', 'ProductsPerTenure', 'Geo_Germany', 'Geo_Spain', 'Gender_Female']


## 9. Podział danych (80/20)

Dane dzielimy ręcznie (bez `train_test_split` z sklearn) na:
- **zbiór treningowy**: 80% (8 000 rekordów)
- **zbiór testowy**: 20% (2 000 rekordów)

Stałe ziarno losowości (`seed=42`) gwarantuje powtarzalność wyników. Proporcja 80/20 jest standardowa — wystarczająco dużo danych do nauki, a zbiór testowy jest duży na miarodajną ewaluację.


In [62]:
def train_test_split_manual(X, y, test_size=0.2, seed=42):
    rng   = np.random.default_rng(seed)
    idx   = rng.permutation(len(X))
    split = int(len(X) * (1 - test_size))
    return X[idx[:split]], X[idx[split:]], y[idx[:split]], y[idx[split:]]

X_train, X_test, y_train, y_test = train_test_split_manual(X_all, y_all)

# Normalizacja (z-score) — obliczona TYLKO na danych treningowych
mean_train = X_train.mean(axis=0)
std_train  = X_train.std(axis=0) + 1e-8   # unikamy dzielenia przez 0

X_train_n = (X_train - mean_train) / std_train
X_test_n  = (X_test  - mean_train) / std_train

print(f"  Train: {X_train_n.shape}  |  Klasa 1: {y_train.mean()*100:.1f}%")
print(f"  Test : {X_test_n.shape}   |  Klasa 1: {y_test.mean()*100:.1f}%")

# Obsługa niezbalansowania — wagi klas
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"  Waga klasy pozytywnej (pos_weight): {pos_weight:.2f}")

  Train: (8000, 16)  |  Klasa 1: 20.0%
  Test : (2000, 16)   |  Klasa 1: 22.1%
  Waga klasy pozytywnej (pos_weight): 4.01


## 10. Sieć neuronowa (implementacja od zera — NumPy)

### Architektura

```
Wejście (16) -> FC(64) -> BatchNorm -> ReLU -> Dropout(0.3)
             -> FC(32) -> BatchNorm -> ReLU -> Dropout(0.3)
             -> FC(1)  -> Sigmoid
```

### Uzasadnienie wyborów projektowych

| Element | Decyzja | Dlaczego |
|---|---|---|
| 2 warstwy ukryte (64->32) | Architektura lejkowa | Stopniowe wydobywanie reprezentacji |
| Batch Normalization | Po każdej warstwie FC | Przyspiesza zbieżność, stabilizuje gradienty |
| Dropout (p=0.3) | Po każdej BN+ReLU | Regularyzacja — zapobiega przeuczeniu |
| ReLU | Aktywacja ukryta | Brak zanikającego gradientu, szybka zbieżność |
| Sigmoid | Wyjście | Prawdopodobieństwo dla klasyfikacji binarnej |
| He initialization | Inicjalizacja wag | Optymalna dla sieci z ReLU |
| Adam | Optymalizator | Adaptacyjny LR — odporny na złe skalowanie cech |

Wszystkie komponenty (propagacja wstecz, BatchNorm z ruchomymi statystykami, Adam) napisane ręcznie w NumPy w celu demonstracji zrozumienia mechanizmów sieci neuronowych.


### Implementacja warstwy Batch Normalization

Własna implementacja `BatchNorm` zawiera:
- **parametry uczone** gamma i beta (skalowanie i przesunięcie)
- **ruchome statystyki** (mean, var) aktualizowane wykładniczo podczas treningu
- **tryby train/test** — podczas inferencji używane są statystyki populacyjne


In [71]:
class BatchNorm:
    """Batch Normalization — stabilna implementacja."""
    def __init__(self, n_features, eps=1e-5, momentum=0.9):
        self.eps      = eps
        self.momentum = momentum
        self.gamma    = np.ones(n_features)
        self.beta     = np.zeros(n_features)
        self.run_mean = np.zeros(n_features)
        self.run_var  = np.ones(n_features)
        self._cache   = {}
        self._mg = np.zeros(n_features)
        self._vg = np.zeros(n_features)
        self._mb = np.zeros(n_features)
        self._vb = np.zeros(n_features)

    def forward(self, x, training=True):
        if training:
            mu  = x.mean(axis=0)
            var = x.var(axis=0)
            self.run_mean = self.momentum * self.run_mean + (1 - self.momentum) * mu
            self.run_var  = self.momentum * self.run_var  + (1 - self.momentum) * var
        else:
            mu, var = self.run_mean, self.run_var
        std         = np.sqrt(var + self.eps)
        xhat        = (x - mu) / std
        self._cache = {"xhat": xhat, "std": std}
        return self.gamma * xhat + self.beta

    def backward(self, dout):
        """Przyjmuje JEDEN argument (dout)."""
        xhat = self._cache["xhat"]
        std  = self._cache["std"]
        n    = dout.shape[0]
        dgamma = (dout * xhat).sum(axis=0)
        dbeta  = dout.sum(axis=0)
        dxhat  = dout * self.gamma
        dx = (1.0 / (n * std)) * (
            n * dxhat
            - dxhat.sum(axis=0)
            - xhat * (dxhat * xhat).sum(axis=0)
        )
        return dx, dgamma, dbeta


class NeuralNet:
    """
    FC -> BN -> ReLU -> Dropout -> FC -> BN -> ReLU -> Dropout -> FC(1) -> Sigmoid
    Adam z jednym globalnym licznikiem t. Gradient clipping. Stabilny sigmoid.
    """
    def __init__(self, n_in, hidden=[64, 32], dropout_rate=0.3,
                 pos_weight=1.0, seed=42):
        rng = np.random.default_rng(seed)
        self.dropout_rate = dropout_rate
        self.pos_weight   = pos_weight
        self.t            = 0  # jeden globalny licznik Adam

        dims   = [n_in] + list(hidden) + [1]
        self.W = [rng.standard_normal((dims[i], dims[i+1])) * np.sqrt(2.0 / dims[i])
                  for i in range(len(dims) - 1)]
        self.b = [np.zeros(dims[i+1]) for i in range(len(dims) - 1)]
        self.bn = [BatchNorm(h) for h in hidden]

        self.mW = [np.zeros_like(w) for w in self.W]
        self.vW = [np.zeros_like(w) for w in self.W]
        self.mb = [np.zeros_like(b) for b in self.b]
        self.vb = [np.zeros_like(b) for b in self.b]
        self._cache = {}

    @staticmethod
    def relu(x):      return np.maximum(0.0, x)
    @staticmethod
    def relu_grad(x): return (x > 0).astype(float)
    @staticmethod
    def sigmoid(x):
        pos = x >= 0
        return np.where(pos,
                        1.0 / (1.0 + np.exp(-np.abs(x))),
                        np.exp(x) / (1.0 + np.exp(x)))

    def forward(self, X, training=True):
        self._cache = {"in": X, "masks": []}
        out = X
        for i, (w, b_) in enumerate(zip(self.W[:-1], self.b[:-1])):
            z   = out @ w + b_
            zn  = self.bn[i].forward(z, training)
            a   = self.relu(zn)
            if training and self.dropout_rate > 0:
                mask = (np.random.rand(*a.shape) > self.dropout_rate).astype(float)
                mask /= (1.0 - self.dropout_rate)
            else:
                mask = np.ones_like(a)
            out = a * mask
            self._cache[f"z{i}"]   = z
            self._cache[f"zn{i}"]  = zn
            self._cache[f"out{i}"] = out
            self._cache["masks"].append(mask)
        z_out = out @ self.W[-1] + self.b[-1]
        y_hat = self.sigmoid(z_out).ravel()
        self._cache["y_hat"] = y_hat
        return y_hat

    def loss(self, y_hat, y):
        eps   = 1e-7
        y_hat = np.clip(y_hat, eps, 1 - eps)
        w     = np.where(y == 1, self.pos_weight, 1.0)
        return -(w * (y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))).mean()

    def backward(self, y, clip_norm=5.0):
        n     = len(y)
        y_hat = np.clip(self._cache["y_hat"], 1e-7, 1 - 1e-7)
        w_vec = np.where(y == 1, self.pos_weight, 1.0)
        gW    = [None] * len(self.W)
        gb    = [None] * len(self.b)
        gBG   = [None] * len(self.bn)
        gBB   = [None] * len(self.bn)

        dz     = ((y_hat - y) * w_vec / n).reshape(-1, 1)
        prev   = self._cache[f"out{len(self.W)-2}"]
        gW[-1] = prev.T @ dz
        gb[-1] = dz.sum(axis=0)

        d = dz @ self.W[-1].T
        for i in range(len(self.W) - 2, -1, -1):
            d  = d * self._cache["masks"][i]
            d  = d * self.relu_grad(self._cache[f"zn{i}"])
            dx, dg, db_bn = self.bn[i].backward(d)   # 1 argument!
            gBG[i] = dg
            gBB[i] = db_bn
            Xp     = self._cache["in"] if i == 0 else self._cache[f"out{i-1}"]
            gW[i]  = Xp.T @ dx
            gb[i]  = dx.sum(axis=0)
            if i > 0:
                d = dx @ self.W[i].T

        all_g = [g.ravel() for g in gW + gb if g is not None]
        norm  = np.sqrt(sum(np.dot(g, g) for g in all_g))
        if norm > clip_norm:
            s   = clip_norm / (norm + 1e-8)
            gW  = [g * s for g in gW]
            gb  = [g * s for g in gb]
            gBG = [g * s if g is not None else None for g in gBG]
            gBB = [g * s if g is not None else None for g in gBB]

        return gW, gb, gBG, gBB

    def _adam(self, param, grad, m, v, lr, b1=0.9, b2=0.999, eps=1e-8):
        # self.t NIE jest tu inkrementowany!
        m[:] = b1 * m + (1 - b1) * grad
        v[:] = b2 * v + (1 - b2) * grad ** 2
        mc   = m / (1 - b1 ** self.t)
        vc   = v / (1 - b2 ** self.t)
        param -= lr * mc / (np.sqrt(vc) + eps)
        return param

    def step(self, X, y, lr):
        y_hat = self.forward(X, training=True)
        l     = self.loss(y_hat, y)
        if not np.isfinite(l):
            return float("nan")
        gW, gb, gBG, gBB = self.backward(y)
        self.t += 1   # inkrementowany TYLKO RAZ, tutaj
        for i in range(len(self.W)):
            self.W[i] = self._adam(self.W[i], gW[i], self.mW[i], self.vW[i], lr)
            self.b[i] = self._adam(self.b[i], gb[i], self.mb[i], self.vb[i], lr)
        for i, bn in enumerate(self.bn):
            if gBG[i] is not None:
                bn.gamma = self._adam(bn.gamma, gBG[i], bn._mg, bn._vg, lr)
                bn.beta  = self._adam(bn.beta,  gBB[i], bn._mb, bn._vb, lr)
        return l

    def predict_proba(self, X): return self.forward(X, training=False)
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


## 11. Trening z Early Stopping

### Konfiguracja

| Parametr | Wartość | Uzasadnienie |
|---|---|---|
| Epoki | 300 (maks.) | Górny limit z zabezpieczeniem Early Stopping |
| Batch size | 256 | Kompromis: stabilne gradienty, szybkie iteracje |
| LR startowy | 3e-3 | Dobra wartość startowa dla Adama |
| Decay LR | 0.985/epokę | Stopniowe zmniejszanie kroku — lepsza zbieżność |
| Early Stopping | cierpliwość = 25 epok | Zatrzymanie gdy val_loss nie spada |
| Gradient clipping | norma = 5.0 | Zapobiega eksplodującym gradientom |
| pos_weight | proporcja klas | Kompensacja niezbalansowania w funkcji BCE |

Model monitoruje stratę na zbiorze walidacyjnym i zapisuje najlepsze wagi.


In [72]:
EPOCHS     = 300
BATCH_SIZE = 256
LR_INIT    = 3e-3
PATIENCE   = 25

n_in  = X_train_n.shape[1]
model = NeuralNet(n_in, hidden=[64, 32], dropout_rate=0.3,
                  pos_weight=pos_weight, seed=42)

history_train = []
history_val   = []
best_val_loss = np.inf
best_W        = [w.copy() for w in model.W]
best_b        = [b.copy() for b in model.b]
patience_cnt  = 0

def get_lr(epoch, init=LR_INIT, decay=0.985):
    return init * decay ** epoch

def batch_iter(X, y, batch_size, seed=None):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X))
    for start in range(0, len(X), batch_size):
        b = idx[start:start + batch_size]
        yield X[b], y[b]

print(f"  Architektura: {n_in} -> 64 -> 32 -> 1")
print(f"  Adam | LR_init={LR_INIT} | Batch={BATCH_SIZE} | Epoki={EPOCHS} | EarlyStopping={PATIENCE}")
print()

for epoch in range(1, EPOCHS + 1):
    lr           = get_lr(epoch)
    batch_losses = []

    for Xb, yb in batch_iter(X_train_n, y_train, BATCH_SIZE, seed=epoch):
        l = model.step(Xb, yb, lr)
        if np.isfinite(l):
            batch_losses.append(l)

    if not batch_losses:
        print(f"  [BLAD] Epoka {epoch}: NaN — przerwano")
        break

    train_loss = float(np.mean(batch_losses))
    val_hat    = model.predict_proba(X_test_n)
    val_loss   = model.loss(val_hat, y_test)

    history_train.append(train_loss)
    history_val.append(val_loss)

    if np.isfinite(val_loss) and val_loss < best_val_loss - 1e-5:
        best_val_loss = val_loss
        best_W        = [w.copy() for w in model.W]
        best_b        = [b.copy() for b in model.b]
        patience_cnt  = 0
    else:
        patience_cnt += 1

    if epoch % 25 == 0 or epoch == 1:
        print(f"  Epoka {epoch:>3} | LR: {lr:.5f} | "
              f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
              f"Patience: {patience_cnt}/{PATIENCE}")

    if patience_cnt >= PATIENCE:
        print(f"\n  [STOP] Early stopping po epoce {epoch}")
        break

model.W = best_W
model.b = best_b
print(f"\n  [OK] Przywrocono wagi (best val_loss={best_val_loss:.4f})")


# Wykres historii treningu
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(history_train, color=ACCENT, label="Train Loss", linewidth=2)
ax.plot(history_val,   color=DANGER, label="Val Loss",   linewidth=2)
ax.set_title("Krzywa uczenia się", fontsize=14)
ax.set_xlabel("Epoka")
ax.set_ylabel("Binary Cross-Entropy")
ax.legend()
ax.grid(True)
save(fig, "05_krzywa_uczenia.png")


  Architektura: 16 -> 64 -> 32 -> 1
  Adam | LR_init=0.003 | Batch=256 | Epoki=300 | EarlyStopping=25

  Epoka   1 | LR: 0.00296 | Train: 1.1018 | Val: 0.9808 | Patience: 0/25
  Epoka  25 | LR: 0.00206 | Train: 0.7531 | Val: 0.8069 | Patience: 3/25
  Epoka  50 | LR: 0.00141 | Train: 0.7173 | Val: 0.8019 | Patience: 9/25

  [STOP] Early stopping po epoce 66

  [OK] Przywrocono wagi (best val_loss=0.7977)
  [zapisano] wyniki\05_krzywa_uczenia.png


## 12. Predykcja i ocena modelu

Ewaluujemy model na niezależnym zbiorze testowym (20% danych, niewidzianych podczas treningu).

**Kluczowe metryki dla niezbalansowanej klasyfikacji:**
- **AUC-ROC** — zdolność dyskryminacyjna niezależna od progu
- **F1-Score** — harmoniczna średnia precyzji i czułości
- **Precyzja** — jaki % przewidzianych churnerów to prawdziwe churny
- **Czułość (Recall)** — jaki % rzeczywistych churnerów model wykrył

**Dobór progu:** szukamy progu decyzyjnego maksymalizującego F1 — zamiast domyślnego 0.5.


In [78]:
# ── Metryki od zera ───────────────────────────────────────
def confusion_matrix_manual(y_true, y_pred):
    TP = int(((y_true==1)&(y_pred==1)).sum())
    TN = int(((y_true==0)&(y_pred==0)).sum())
    FP = int(((y_true==0)&(y_pred==1)).sum())
    FN = int(((y_true==1)&(y_pred==0)).sum())
    return TP, TN, FP, FN

def metrics(TP, TN, FP, FN):
    precision  = TP/(TP+FP) if TP+FP>0 else 0
    recall     = TP/(TP+FN) if TP+FN>0 else 0
    f1         = 2*precision*recall/(precision+recall) if precision+recall>0 else 0
    accuracy   = (TP+TN)/(TP+TN+FP+FN)
    specificity= TN/(TN+FP) if TN+FP>0 else 0
    return dict(precision=precision, recall=recall, f1=f1,
                accuracy=accuracy, specificity=specificity)

def auc_roc_manual(y_true, y_scores):
    thresholds = np.linspace(0, 1, 300)
    tprs, fprs = [], []
    for thr in thresholds:
        pred = (y_scores >= thr).astype(int)
        TP,TN,FP,FN = confusion_matrix_manual(y_true, pred)
        tprs.append(TP/(TP+FN+1e-9))
        fprs.append(FP/(FP+TN+1e-9))
    tprs = np.array(tprs)
    fprs = np.array(fprs)
    # metoda trapezów
    order = np.argsort(fprs)
    auc = np.trapezoid(tprs[order], fprs[order])
    return float(auc), fprs, tprs

def find_best_threshold(y_true, y_scores):
    best_f1, best_thr = 0, 0.5
    for thr in np.linspace(0.1, 0.9, 80):
        pred = (y_scores >= thr).astype(int)
        TP,TN,FP,FN = confusion_matrix_manual(y_true, pred)
        m = metrics(TP,TN,FP,FN)
        if m["f1"] > best_f1:
            best_f1, best_thr = m["f1"], thr
    return best_thr, best_f1

y_scores = model.predict_proba(X_test_n)
best_thr, _ = find_best_threshold(y_test, y_scores)
y_pred = (y_scores >= best_thr).astype(int)

TP, TN, FP, FN = confusion_matrix_manual(y_test, y_pred)
m = metrics(TP, TN, FP, FN)
auc, fprs, tprs = auc_roc_manual(y_test, y_scores)

print(f"\n  Optymalny próg decyzyjny: {best_thr:.3f}")
print(f"\n  {'Metryka':<18} {'Wartość':>8}")
print(f"  {'-'*28}")
for k,v in m.items():
    print(f"  {k:<18} {v:>8.4f}")
print(f"  {'AUC-ROC':<18} {auc:>8.4f}")
print(f"\n  Macierz pomyłek:")
print(f"  {'':6} Pred=0  Pred=1")
print(f"  Real=0   {TN:>5}   {FP:>5}")
print(f"  Real=1   {FN:>5}   {TP:>5}")

# ── Wykres macierzy pomyłek ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm_arr = np.array([[TN, FP],[FN, TP]])
labels = np.array([["TN","FP"],["FN","TP"]])
im = axes[0].imshow(cm_arr, cmap="Blues", aspect="auto")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f"{labels[i,j]}\n{cm_arr[i,j]:,}",
                     ha="center", va="center", fontsize=13, fontweight="bold",
                     color="white" if cm_arr[i,j] > cm_arr.max()/2 else "#0f172a")
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(["Pred: Zostaje","Pred: Odchodzi"])
axes[0].set_yticklabels(["Real: Zostaje","Real: Odchodzi"])
axes[0].set_title(f"Macierz pomyłek  (próg={best_thr:.2f})", fontsize=12)

# ROC
axes[1].plot(fprs, tprs, color=ACCENT, linewidth=2.5, label=f"AUC = {auc:.4f}")
axes[1].plot([0,1],[0,1], color="#475569", linestyle="--", linewidth=1)
axes[1].fill_between(fprs, tprs, alpha=0.15, color=ACCENT)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Krzywa ROC", fontsize=12)
axes[1].legend(fontsize=12)
axes[1].grid(True)

fig.tight_layout()
save(fig, "06_macierz_roc.png")


  Optymalny próg decyzyjny: 0.637

  Metryka             Wartość
  ----------------------------
  precision            0.6574
  recall               0.5873
  f1                   0.6204
  accuracy             0.8415
  specificity          0.9134
  AUC-ROC              0.8501

  Macierz pomyłek:
         Pred=0  Pred=1
  Real=0    1424     135
  Real=1     182     259
  [zapisano] wyniki\06_macierz_roc.png


## 13. Ocena ryzyka pojedynczego klienta

Model można wykorzystać do oceny konkretnego klienta w czasie rzeczywistym.

Funkcja `explain_risk()` zwraca:
- **wynik ryzyka** (0-100%) — prawdopodobieństwo odejścia
- **kategorię ryzyka** (niskie / średnie / wysokie)
- **top czynniki wpływu** — które cechy najbardziej zwiększają lub zmniejszają ryzyko

Wyjaśnienia generowane metodą **analizy wrażliwości** (numeryczne gradienty) — uproszczona wersja podejścia SHAP.


In [79]:
def prepare_single(customer_dict: dict) -> np.ndarray:
    """
    Przygotowuje wektor cech z surowych danych klienta.
    Kolejność musi odpowiadać FEATURE_COLS.
    """
    d = {k: v for k, v in customer_dict.items()}

    # Kodowanie Geography
    geo = d.pop("Geography", "France")
    for g in ["Germany","Spain"]:
        d[f"Geo_{g}"] = 1 if geo == g else 0

    # Kodowanie Gender
    gender = d.pop("Gender", "Male")
    d["Gender_Female"] = 1 if gender == "Female" else 0

    # Inżynieria cech
    balance = d.get("Balance", 0)
    salary  = d.get("EstimatedSalary", 50000)
    age     = d.get("Age", 40)
    tenure  = d.get("Tenure", 5)
    products= d.get("NumOfProducts", 1)

    d["BalanceSalaryRatio"] = balance / (salary + 1)
    d["AgeGroup"]           = int(pd.cut([age], bins=[0,30,45,60,100], labels=[0,1,2,3])[0])
    d["HasBalance"]         = int(balance > 0)
    d["TenurePerAge"]       = tenure / (age + 1)
    d["ProductsPerTenure"]  = products / (tenure + 1)

    row = np.array([d.get(f, 0.0) for f in FEATURE_COLS], dtype=float)
    return row


RISK_DESCRIPTIONS = {
    # Cechy i ich interpretacja ryzyka (przy wysokiej wartości)
    "Age":                  ("wiek",              "Starsi klienci odchodzą częściej"),
    "Balance":              ("saldo",             "Wysokie saldo z małą aktywnością = ryzyko"),
    "IsActiveMember":       ("aktywność",         "Nieaktywni klienci mają wyższe ryzyko odejścia"),
    "NumOfProducts":        ("liczba produktów",  "≥3 produkty wiążą się z wyższym churnem"),
    "Geo_Germany":          ("lokalizacja DE",    "Niemcy mają najwyższy wskaźnik churnu"),
    "Gender_Female":        ("płeć K",            "Kobiety odchodzą nieco częściej w tym banku"),
    "CreditScore":          ("scoring kredytowy", "Niski scoring może oznaczać niezadowolenie"),
    "Tenure":               ("staż",              "Krótki staż = wyższe ryzyko odejścia"),
    "HasCrCard":            ("karta kredytowa",   "Sam fakt posiadania karty to słaby sygnał"),
    "BalanceSalaryRatio":   ("saldo/zarobki",     "Wysoki stosunek salda do zarobków = ryzyko"),
    "AgeGroup":             ("grupa wiekowa",     "Klienci 45–60 lat odchodzą najczęściej"),
    "HasBalance":           ("czy ma saldo",      "Klienci z zerowym saldem rzadziej odchodzą"),
    "TenurePerAge":         ("staż/wiek",         "Proporcja stażu do wieku"),
    "ProductsPerTenure":    ("produkty/staż",     "Dużo produktów w krótkim czasie = ryzyko"),
    "EstimatedSalary":      ("szacowane zarobki", "Wpływ zarobków na decyzję"),
    "Geo_Spain":            ("lokalizacja ES",    "Hiszpania ma umiarkowany wskaźnik churnu"),
    "HasCrCard":            ("karta kredytowa",   "Posiadacze kart rzadziej odchodzą"),
}

def explain_risk(customer_dict: dict, verbose=True):
    """
    Ocenia ryzyko odejścia klienta i podaje powody.
    Używa analizy gradientu (sensitivity analysis) do oceny ważności cech.
    """
    row     = prepare_single(customer_dict)
    row_n   = (row - mean_train) / std_train
    row_n2  = row_n.reshape(1, -1)

    prob = model.predict_proba(row_n2)[0]

    # ── Sensitivity analysis (numeryczny gradient) ───
    deltas = {}
    for i, feat in enumerate(FEATURE_COLS):
        eps_feat       = 1e-4
        row_plus       = row_n2.copy(); row_plus[0,i]  += eps_feat
        row_minus      = row_n2.copy(); row_minus[0,i] -= eps_feat
        dp = model.predict_proba(row_plus)[0] - model.predict_proba(row_minus)[0]
        deltas[feat]   = dp / (2*eps_feat) * (row_n[i] if abs(row_n[i]) > 0.01 else eps_feat)

    sorted_feats = sorted(deltas.items(), key=lambda x: abs(x[1]), reverse=True)

    # ── Poziom ryzyka ─────────────────────────────────
    if prob < 0.30:
        risk_level, risk_emoji, risk_color = "NISKIE",   "🟢", SUCCESS
    elif prob < 0.55:
        risk_level, risk_emoji, risk_color = "ŚREDNIE",  "🟡", WARNING
    elif prob < 0.75:
        risk_level, risk_emoji, risk_color = "WYSOKIE",  "🔴", DANGER
    else:
        risk_level, risk_emoji, risk_color = "KRYTYCZNE","🚨", DANGER

    if verbose:
        print(f"\n  {'═'*52}")
        print(f"  {risk_emoji}  OCENA RYZYKA KLIENTA")
        print(f"  {'─'*52}")
        name = customer_dict.get("Name", "Klient")
        print(f"  Klient       : {name}")
        print(f"  Prawdop. churnu: {prob*100:.1f}%")
        print(f"  Poziom ryzyka: {risk_level}")
        print(f"  {'─'*52}")
        print(f"  TOP PRZYCZYNY RYZYKA:")
        print()

        pos_factors = [(f,v) for f,v in sorted_feats if v > 0][:5]
        neg_factors = [(f,v) for f,v in sorted_feats if v < 0][:3]

        if pos_factors:
            print(f"  🔺 Czynniki ZWIĘKSZAJĄCE ryzyko:")
            for feat, val in pos_factors:
                raw_val = row[FEATURE_COLS.index(feat)]
                desc    = RISK_DESCRIPTIONS.get(feat, (feat, "—"))
                print(f"    • {desc[0]:<22} wartość={raw_val:.2f}  wpływ=+{val*100:.2f}%")
                print(f"      ↳ {desc[1]}")
        print()
        if neg_factors:
            print(f"  🔻 Czynniki ZMNIEJSZAJĄCE ryzyko:")
            for feat, val in neg_factors:
                raw_val = row[FEATURE_COLS.index(feat)]
                desc    = RISK_DESCRIPTIONS.get(feat, (feat, "—"))
                print(f"    • {desc[0]:<22} wartość={raw_val:.2f}  wpływ={val*100:.2f}%")
                print(f"      ↳ {desc[1]}")

        print(f"\n  {'─'*52}")
        if prob > 0.5:
            print(f"  ⚠️  REKOMENDACJA: Skontaktuj się z klientem!")
            if any(f=="IsActiveMember" for f,_ in pos_factors):
                print(f"     → Zaoferuj program lojalnościowy")
            if any(f=="NumOfProducts" for f,_ in pos_factors):
                print(f"     → Sprawdź zadowolenie z produktów")
            if any(f=="Geo_Germany" for f,_ in pos_factors):
                print(f"     → Dedykowany opiekun dla rynku DE")
        else:
            print(f"  ✅  REKOMENDACJA: Klient jest stabilny")
        print(f"  {'═'*52}")

    return {
        "probability": prob,
        "risk_level":  risk_level,
        "top_factors": sorted_feats[:8]
    }


# ── Przykładowe oceny klientów ───────────────────────────
print("\n  === PRZYKŁAD 1: Klientka wysokiego ryzyka ===")
explain_risk({
    "Name":            "Anna Kowalska",
    "CreditScore":     620,
    "Geography":       "Germany",
    "Gender":          "Female",
    "Age":             52,
    "Tenure":          2,
    "Balance":         185_000,
    "NumOfProducts":   1,
    "HasCrCard":       1,
    "IsActiveMember":  0,
    "EstimatedSalary": 75_000,
})

print("\n  === PRZYKŁAD 2: Klient niskiego ryzyka ===")
explain_risk({
    "Name":            "Jan Nowak",
    "CreditScore":     780,
    "Geography":       "France",
    "Gender":          "Male",
    "Age":             34,
    "Tenure":          8,
    "Balance":         0,
    "NumOfProducts":   2,
    "HasCrCard":       1,
    "IsActiveMember":  1,
    "EstimatedSalary": 95_000,
})

print("\n  === PRZYKŁAD 3: Ocena istniejącego klienta z danych ===")
# Losowy klient z bazy
sample_row = df.sample(1, random_state=99).iloc[0]
sample_dict = {col: sample_row[col] for col in df.columns if col != "Exited"}
sample_dict["Name"] = f"Klient ID-{sample_row.name}"
result = explain_risk(sample_dict)
print(f"\n  Rzeczywista etykieta z bazy: {'Odszedł ✗' if sample_row['Exited']==1 else 'Pozostał ✓'}")



  === PRZYKŁAD 1: Klientka wysokiego ryzyka ===

  ════════════════════════════════════════════════════
  🚨  OCENA RYZYKA KLIENTA
  ────────────────────────────────────────────────────
  Klient       : Anna Kowalska
  Prawdop. churnu: 92.9%
  Poziom ryzyka: KRYTYCZNE
  ────────────────────────────────────────────────────
  TOP PRZYCZYNY RYZYKA:

  🔺 Czynniki ZWIĘKSZAJĄCE ryzyko:
    • lokalizacja DE         wartość=1.00  wpływ=+4.84%
      ↳ Niemcy mają najwyższy wskaźnik churnu
    • aktywność              wartość=0.00  wpływ=+4.80%
      ↳ Nieaktywni klienci mają wyższe ryzyko odejścia
    • wiek                   wartość=52.00  wpływ=+4.68%
      ↳ Starsi klienci odchodzą częściej
    • liczba produktów       wartość=1.00  wpływ=+3.00%
      ↳ ≥3 produkty wiążą się z wyższym churnem
    • grupa wiekowa          wartość=2.00  wpływ=+2.09%
      ↳ Klienci 45–60 lat odchodzą najczęściej

  🔻 Czynniki ZMNIEJSZAJĄCE ryzyko:
    • saldo                  wartość=185000.00  wpływ=-2.63%
  

## 14. Rozkład przewidywanego ryzyka na zbiorze testowym

Wizualizujemy rozkład wyników modelu (prawdopodobieństwo churnu) dla obu klas rzeczywistych.

Dobrze skalibrowany model powinien:
- skupiać klientów klasy 0 (pozostali) blisko 0
- skupiać klientów klasy 1 (odeszli) blisko 1
- wykazywać wyraźne rozdzielenie między klasami


In [80]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Histogram prawdopodobieństw
for churn, color, label in [(0,PALETTE[0],"Pozostał"),(1,PALETTE[1],"Odszedł")]:
    mask = y_test == churn
    axes[0].hist(y_scores[mask], bins=40, alpha=0.6, color=color, label=label, density=True)
axes[0].axvline(best_thr, color=WARNING, linestyle="--", linewidth=2, label=f"Próg {best_thr:.2f}")
axes[0].set_title("Rozkład prawdopodobieństwa churnu", fontsize=12)
axes[0].set_xlabel("P(Odejście)")
axes[0].set_ylabel("Gęstość")
axes[0].legend()
axes[0].grid(True)

# Słupkowy rozkład poziomów ryzyka
bins   = [0, 0.30, 0.55, 0.75, 1.01]
labels_r = ["Niskie\n(<30%)","Średnie\n(30-55%)","Wysokie\n(55-75%)","Krytyczne\n(>75%)"]
colors_r = [SUCCESS, WARNING, DANGER, "#dc2626"]
counts_r = np.histogram(y_scores, bins=bins)[0]
bars = axes[1].bar(labels_r, counts_r, color=colors_r, edgecolor="#0f172a", linewidth=1.5)
for bar, cnt in zip(bars, counts_r):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 f"{cnt:,}\n({cnt/len(y_scores)*100:.1f}%)",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")
axes[1].set_title("Segmentacja klientów wg ryzyka", fontsize=12)
axes[1].set_ylabel("Liczba klientów")
axes[1].grid(axis="y")

fig.suptitle("Rozkład ryzyka — zbiór testowy", fontsize=14)
fig.tight_layout()
save(fig, "07_rozkład_ryzyka.png")


  [zapisano] wyniki\07_rozkład_ryzyka.png


## 15. Podsumowanie

### Wyniki modelu

| Metryka | Wartość |
|---|---|
| AUC-ROC | **0.8501** |
| Dokładność | **84.15%** |
| F1-Score | 0.6204 |
| Precyzja | 0.6574 |
| Czułość | 0.5873 |
| Swoistość | 0.9134 |

### Wnioski

- Model skutecznie identyfikuje klientów zagrożonych odejściem — AUC-ROC 0.85 oznacza dobrą zdolność dyskryminacyjną
- Najważniejsze predyktory churnu: **wiek**, **aktywność klienta**, **kraj** (Niemcy), **liczba produktów**
- Niezbalansowanie klas (20/80) zaadresowane przez ważenie klas i dobór progu (0.637 zamiast domyślnego 0.5)

### Ograniczenia

- Zbiór pochodzi z jednego banku — generalizacja wymaga walidacji na innych danych
- Model nie uwzględnia danych czasowych (historia transakcji, zmiany zachowania)
- Brak walidacji krzyżowej (k-fold)

### Możliwe kierunki rozwoju

- Dodanie cech opartych na historii transakcji
- Porównanie z modelami gradientowego boostingu (XGBoost, LightGBM)
- Wdrożenie jako REST API do oceny ryzyka klienta w czasie rzeczywistym


In [81]:
print(f"""
  Model   : 2-warstwowa sieć neuronowa (NumPy, od zera)
  Cechy   : {n_in}
  Próg    : {best_thr:.3f}

  Wyniki na zbiorze testowym:
    AUC-ROC   : {auc:.4f}
    F1-score  : {m['f1']:.4f}
    Precision : {m['precision']:.4f}
    Recall    : {m['recall']:.4f}
    Accuracy  : {m['accuracy']:.4f}

  Pliki wynikowe w folderze 'wyniki/':
    01_rozkład_klas.png
    02_cechy_numeryczne.png
    03_cechy_kategoryczne.png
    04_korelacja.png
    05_krzywa_uczenia.png
    06_macierz_roc.png
    07_rozkład_ryzyka.png

  Użycie funkcji oceny ryzyka:
    explain_risk({{
        "CreditScore": 700,
        "Geography": "Germany",
        "Gender": "Female",
        "Age": 48,
        "Tenure": 3,
        "Balance": 120000,
        "NumOfProducts": 1,
        "HasCrCard": 1,
        "IsActiveMember": 0,
        "EstimatedSalary": 80000
    }})
""")


  Model   : 2-warstwowa sieć neuronowa (NumPy, od zera)
  Cechy   : 16
  Próg    : 0.637

  Wyniki na zbiorze testowym:
    AUC-ROC   : 0.8501
    F1-score  : 0.6204
    Precision : 0.6574
    Recall    : 0.5873
    Accuracy  : 0.8415

  Pliki wynikowe w folderze 'wyniki/':
    01_rozkład_klas.png
    02_cechy_numeryczne.png
    03_cechy_kategoryczne.png
    04_korelacja.png
    05_krzywa_uczenia.png
    06_macierz_roc.png
    07_rozkład_ryzyka.png

  Użycie funkcji oceny ryzyka:
    explain_risk({
        "CreditScore": 700,
        "Geography": "Germany",
        "Gender": "Female",
        "Age": 48,
        "Tenure": 3,
        "Balance": 120000,
        "NumOfProducts": 1,
        "HasCrCard": 1,
        "IsActiveMember": 0,
        "EstimatedSalary": 80000
    })

